# Retention Policy Salary-Allocation Equity Audit

This executed notebook audits whether the frozen expected-value policy directs supportive human-review resources disproportionately toward higher-paid employees.

> All employees, salaries, probabilities, and financial values are synthetic. This audit is descriptive, does not access outcome columns, does not change the frozen policy, and does not authorize automatic employment action.

## Audit question

The policy multiplies calibrated attrition probability by a salary-based replacement-cost estimate. The audit measures salary concentration directly and compares the frozen policy with probability-only, salary-capped, and constant-cost sensitivity policies.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "policy_equity"
print(f"Loaded aggregate policy-equity outputs from {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Loaded aggregate policy-equity outputs from data/processed/policy_equity


## Headline salary concentration

The selected population has a higher mean salary in validation, final test, and current scoring. A review flag is descriptive evidence of a policy mechanism, not a legal fairness verdict.

In [2]:
summary = pd.read_csv(OUTPUT_DIR / "policy_equity_population_summary.csv")
columns = [
    "period", "population_rows", "selected_count",
    "population_mean_salary_usd", "selected_mean_salary_usd",
    "selected_to_population_mean_salary_ratio",
    "lowest_salary_quintile_selection_rate",
    "highest_salary_quintile_selection_rate",
    "highest_to_lowest_selection_rate_ratio",
    "overlap_with_probability_top_700", "review_flag",
]
print(summary[columns].to_string(index=False))

         period  population_rows  selected_count  population_mean_salary_usd  selected_mean_salary_usd  selected_to_population_mean_salary_ratio  lowest_salary_quintile_selection_rate  highest_salary_quintile_selection_rate  highest_to_lowest_selection_rate_ratio  overlap_with_probability_top_700  review_flag
2024 validation             5521             700                92809.726499             121964.428571                                  1.314134                               0.007240                                0.325181                               44.915648                               212         True
2025 final test             6641             700                94466.872459             106393.142857                                  1.126248                               0.012792                                0.142319                               11.126019                               323         True
   2026 current             7305             700                957

## Current policy sensitivity

The salary-capped alternative reduces selected salary concentration while preserving more of the original modeled financial value than probability-only selection. These alternatives are not adopted policies.

In [3]:
comparison = pd.read_csv(OUTPUT_DIR / "policy_equity_policy_comparison.csv")
current = comparison[comparison["period"].eq("2026 current")]
columns = [
    "policy", "selected_count", "selected_mean_salary_usd",
    "selected_mean_probability",
    "predicted_net_value_under_original_economics_usd",
    "overlap_with_frozen_policy",
]
print(current[columns].to_string(index=False))

                      policy  selected_count  selected_mean_salary_usd  selected_mean_probability  predicted_net_value_under_original_economics_usd  overlap_with_frozen_policy
Frozen expected-value policy             700             125669.714286                   0.200196                                      2.396216e+06                         700
         Top 700 probability             700              82995.000000                   0.237492                                      1.666616e+06                         321
Salary-capped expected value             700             110734.285714                   0.212603                                      2.251288e+06                         568
   Constant replacement cost             700              82995.000000                   0.237492                                      1.666616e+06                         321


## Manufacturing and validation evidence

Manufacturing's 162 selected employees average $108,935 versus $88,464 for all eligible Manufacturing employees. All audit checks pass without reading outcomes or changing employee selections.

In [4]:
department = pd.read_csv(OUTPUT_DIR / "policy_equity_department_summary.csv")
manufacturing = department[
    department["period"].eq("2026 current")
    & department["department_name"].eq("Manufacturing")
].iloc[0]
validation = pd.read_csv(OUTPUT_DIR / "policy_equity_validation.csv")
print(f"Manufacturing selected employees: {int(manufacturing['selected_employees'])}")
print(f"Manufacturing eligible mean salary: ${manufacturing['population_mean_salary_usd']:,.0f}")
print(f"Manufacturing selected mean salary: ${manufacturing['selected_mean_salary_usd']:,.0f}")
print(f"Validation checks passed: {(validation['status'] == 'PASS').sum()}/{len(validation)}")

Manufacturing selected employees: 162
Manufacturing eligible mean salary: $88,464
Manufacturing selected mean salary: $108,935
Validation checks passed: 12/12


## Visual evidence

The committed figures show selected salary across policies, selection rates by current salary quintile, and the modeled value-versus-salary-composition tradeoff.

In [5]:
figure_dir = OUTPUT_DIR / "figures"
for filename in [
    "policy_selected_salary_comparison.png",
    "current_pay_quintile_selection.png",
    "policy_value_equity_tradeoff.png",
]:
    display(Image(filename=str(figure_dir / filename)))

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

## Governance conclusion

The frozen expected-value policy is financially optimized, not equity-neutral. Salary-capped and salary-neutral policies are sensitivity analyses only. Selecting a replacement policy would require a new holdout or prospective evaluation; the once-tested Version 2 policy is not retuned here.